# Diagnostics — test sylwetki, wyłącznie inferencja GPU

**Nie trenujemy modelu ponownie.** Zeszyt uruchamia dokładnie zamrożony skrypt, plan i checkpoint YOLO11l-cls, seed 0.

1. W Colab wybierz **Środowisko wykonawcze → Zmień typ środowiska → GPU**. Do najwierniejszego odtworzenia wcześniejszego środowiska zalecany jest **A100**.
2. Wybierz **Uruchom wszystko**.
3. Gdy pojawi się przycisk, wgraj **jeden plik `PRIVATE_Diagnostics_shape_test_GPU.zip`**. Zawiera komplet materiałów potrzebnych do tego testu, w tym 150 oryginalnych PNG i zamrożone wagi. To pakiet prywatny — nie publikuj go na GitHub/Zenodo.
4. Na końcu pobierz ZIP z wynikami i prześlij go do dalszej oceny.

Najpierw musi przejść kontrola odtworzenia 150 klasyfikacji, 138 poprawnych i maksymalnej różnicy prawdopodobieństw ≤ 0.0001. Dopiero wtedy skrypt wykonuje dwa zaplanowane warianty: jednolita szarość 128 oraz średnia jasność danego PNG. Zachowuje oryginalne wymiary i ten sam resize/padding.

**Informacja o dotychczasowej próbie:** kontrola CPU odtworzyła wszystkie 150 decyzji, lecz nie spełniła zamrożonej tolerancji prawdopodobieństw; dlatego nie uruchomiono wariantów sylwetki. Ten zeszyt ponawia wykonanie w środowisku GPU, bez zmiany tolerancji, danych, planu ani skryptu.

Test jest dodatkową analizą post hoc obrazów spoza rozkładu treningowego. Niska skuteczność na sylwetkach nie dowodzi wykorzystania wyłącznie kości ani braku shortcut learning.


## 1. Sprawdzenie GPU i instalacja zapisanych wersji

Instalacja może potrwać kilka minut. Inferencja będzie uruchomiona w nowym procesie Python, więc nie korzysta z wcześniej zaimportowanych wersji bibliotek w tym zeszycie.


In [ ]:
import sys, subprocess, importlib.metadata, json, pathlib, os, datetime, hashlib, shutil, zipfile, stat

gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("Brak GPU. Włącz GPU w ustawieniach środowiska Colab.")
print("GPU:", gpu_check.stdout.strip())

def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

if version("torch") != "2.11.0+cu128" or version("torchvision") != "0.26.0+cu128":
    subprocess.run([sys.executable, "-m", "pip", "install", "--index-url", "https://download.pytorch.org/whl/cu128", "torch==2.11.0+cu128", "torchvision==0.26.0+cu128"], check=True)
other = ["ultralytics==8.4.163", "pillow==11.3.0", "numpy==2.2.6", "scipy==1.15.3", "scikit-learn==1.8.0", "pandas==2.2.3", "matplotlib==3.10.7", "opencv-python==5.0.0.93"]
if any(version(spec.split("==")[0]) != spec.split("==")[1] for spec in other):
    subprocess.run([sys.executable, "-m", "pip", "install", *other], check=True)
assert version("torch") == "2.11.0+cu128", "Nieprawidłowa wersja CUDA pakietu torch"
assert version("torchvision") == "0.26.0+cu128", "Nieprawidłowa wersja CUDA pakietu torchvision"
print({name: version(name) for name in ["torch", "torchvision", "ultralytics", "Pillow", "numpy"]})


GPU: NVIDIA A100-SXM4-40GB, 40960 MiB
{'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'ultralytics': '8.4.163', 'Pillow': '11.3.0', 'numpy': '2.2.6'}


## 2. Wgraj jeden prywatny pakiet ZIP

Nie trzeba wskazywać folderów na Drive ani osobno rozpakowywać bazy lub wag. Oryginalne pliki pozostaną niezmienione.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
work = Path("/content") / ("Diagnostics_shape_test_" + stamp)
work.mkdir(exist_ok=False)
preuploaded = Path("/content/PRIVATE_Diagnostics_shape_test_GPU.zip")
if not preuploaded.is_file():
    raise RuntimeError("Upload the unchanged input ZIP using the Colab Files panel first.")
uploaded = {preuploaded.name: preuploaded.read_bytes()}
if len(uploaded) != 1:
    raise RuntimeError("Wgraj dokładnie jeden plik PRIVATE_Diagnostics_shape_test_GPU.zip.")
name, archive_bytes = next(iter(uploaded.items()))
archive_path = work / Path(name).name
archive_path.write_bytes(archive_bytes)
archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
del uploaded, archive_bytes
print("Wgrano:", archive_path.name, "SHA256:", archive_sha256)

unpacked = work / "package"
unpacked.mkdir()
with zipfile.ZipFile(archive_path) as archive:
    infos = archive.infolist()
    if len(infos) > 2000 or sum(info.file_size for info in infos) > 512*1024*1024:
        raise RuntimeError("Niespodziewanie duży pakiet — sprawdź wybrany plik.")
    names = set()
    for info in infos:
        path = PurePosixPath(info.filename)
        if path.is_absolute() or ".." in path.parts or "\\" in info.filename or stat.S_ISLNK(info.external_attr >> 16):
            raise RuntimeError("Niebezpieczna ścieżka lub dowiązanie w ZIP.")
        if info.filename in names:
            raise RuntimeError("Powtórzona ścieżka w ZIP.")
        names.add(info.filename)
    archive.extractall(unpacked)

scripts = list(unpacked.rglob("run_shape_sensitivity.py"))
if len(scripts) != 1:
    raise RuntimeError("Pakiet musi zawierać dokładnie jeden zamrożony skrypt analizy.")
script = scripts[0]
bundle = script.parent
project = bundle / "reference_project"
plan = bundle / "analysis_plan.json"
freeze = bundle / "analysis_freeze.json"
data_root = bundle / "dataset"
checkpoint = bundle / "checkpoint" / "selected.pt"
for path in [plan, freeze, bundle / "PLAN_ANALIZY.txt", checkpoint, project / "data/clean_manifest.csv"]:
    if not path.is_file(): raise RuntimeError(f"Brak wymaganego pliku: {path.name}")
if not (data_root / "test").is_dir(): raise RuntimeError("Brak dataset/test w pakiecie.")

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""): h.update(block)
    return h.hexdigest()
lock = json.loads(freeze.read_text())
if file_sha256(script) != lock["script_sha256"] or file_sha256(plan) != lock["plan_sha256"]:
    raise RuntimeError("Kod lub plan nie zgadza się z zamrożonymi hashami.")
if file_sha256(bundle / "PLAN_ANALIZY.txt") != lock["readable_plan_sha256"]:
    raise RuntimeError("Opis planu zmienił się po zamrożeniu.")
for relative, expected in lock["source_hashes"].items():
    if file_sha256(project / relative) != expected:
        raise RuntimeError(f"Niezgodny plik referencyjny: {relative}")
if file_sha256(checkpoint) != lock["configuration"]["checkpoint_sha256"]:
    raise RuntimeError("Niezgodne wagi modelu.")
print("Pakiet i zamrożone materiały sprawdzone. Skrypt przed inferencją dodatkowo sprawdzi wszystkie 150 PNG.")


Wgrano: PRIVATE_Diagnostics_shape_test_GPU.zip SHA256: a8f6ca4fa97513ce07334452143ffe61668ee6af696f7d3b08b52763abc49c2e
Pakiet i zamrożone materiały sprawdzone. Skrypt przed inferencją dodatkowo sprawdzi wszystkie 150 PNG.


## 3. Kontrola odtworzenia i dwa zamrożone warianty

Nowy proces ustawia flagi zapisane podczas wcześniejszego, dokładnego odtworzenia GPU dla Eigen-CAM. Nie zmienia skryptu, planu, wag ani tolerancji. Nawet na GPU wynik musi przejść tę samą kontrolę — typ karty sam w sobie nie gwarantuje zgodności.

Jeżeli kontrola zakończy się błędem, zeszyt zatrzyma analizę wariantów i nadal przygotuje ZIP z diagnostyką. Nie poprawiaj tolerancji i nie uruchamiaj nowego treningu.


In [ ]:
output = work / "results"
log_path = work / "execution.log"
runtime_audit = work / "gpu_runtime_target_and_observed.json"
command = [str(script), "run", "--project", str(project), "--plan", str(plan), "--freeze", str(freeze),
           "--data-root", str(data_root), "--checkpoint", str(checkpoint), "--output", str(output),
           "--device", "cuda:0", "--workdir", "/content"]
wrapper = r'''import os, sys, json, runpy, platform
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
if not torch.cuda.is_available() or torch.version.cuda != "12.8":
    raise RuntimeError("Expected CUDA 12.8 GPU runtime; inspect pinned Torch installation.")
torch.use_deterministic_algorithms(False)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("highest")
audit = {"purpose":"Target numerical flags from previous exact A100 Eigen-CAM replay; current environment recorded before this retry.",
         "python":sys.version, "torch":torch.__version__, "torch_cuda":torch.version.cuda,
         "cudnn_version":torch.backends.cudnn.version(), "gpu":torch.cuda.get_device_name(0),
         "gpu_capability":list(torch.cuda.get_device_capability(0)),
         "deterministic_algorithms":torch.are_deterministic_algorithms_enabled(),
         "cudnn_benchmark":torch.backends.cudnn.benchmark, "cudnn_deterministic":torch.backends.cudnn.deterministic,
         "cuda_matmul_allow_tf32":torch.backends.cuda.matmul.allow_tf32, "cudnn_allow_tf32":torch.backends.cudnn.allow_tf32,
         "float32_matmul_precision":torch.get_float32_matmul_precision(), "cublas_workspace_config":os.environ["CUBLAS_WORKSPACE_CONFIG"],
         "historical_exact_replay_gpu":"NVIDIA A100-SXM4-80GB", "historical_exact_replay_cudnn_version":91900}
open(sys.argv[2],"w").write(json.dumps(audit,indent=2)+"\n")
print(json.dumps(audit,indent=2),flush=True)
sys.argv = json.loads(sys.argv[1])
runpy.run_path(sys.argv[0], run_name="__main__")
'''
with log_path.open("w") as log:
    process = subprocess.Popen([sys.executable, "-u", "-c", wrapper, json.dumps(command), str(runtime_audit)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    return_code = process.wait()

output.mkdir(exist_ok=True)
shutil.copy2(log_path, output / log_path.name)
if runtime_audit.is_file(): shutil.copy2(runtime_audit, output / runtime_audit.name)
receipt = {"return_code":return_code, "input_package_name":archive_path.name, "input_package_sha256":archive_sha256,
           "frozen_script_sha256":file_sha256(script), "freeze_sha256":file_sha256(freeze),
           "cpu_attempt_preceded_this_gpu_retry":True, "analysis_tolerance_changed":False}
(output / "execution_receipt.json").write_text(json.dumps(receipt,indent=2)+"\n")
if return_code != 0:
    print("ANALIZA NIEUKOŃCZONA: kontrola lub wykonanie zatrzymały test. Pobierz ZIP z diagnostyką z kolejnej komórki.")
else:
    summary = json.loads((output / "summary.json").read_text())
    if summary["status"] != "COMPLETE": raise RuntimeError("Brak potwierdzenia ukończenia analizy.")
    for condition, result in summary["conditions"].items():
        a = result["accuracy"]
        print(f'{condition}: {a["correct"]}/{a["total"]}; accuracy {100*a["estimate"]:.1f}%; '
              f'Wilson 95% CI {100*a["ci_low"]:.1f}–{100*a["ci_high"]:.1f}%; '
              f'balanced accuracy {100*result["balanced_accuracy"]:.1f}%; macro-F1 {result["macro_f1"]:.3f}')


{
  "purpose": "Target numerical flags from previous exact A100 Eigen-CAM replay; current environment recorded before this retry.",
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "torch": "2.11.0+cu128",
  "torch_cuda": "12.8",
  "cudnn_version": 91900,
  "gpu": "NVIDIA A100-SXM4-40GB",
  "gpu_capability": [
    8,
    0
  ],
  "deterministic_algorithms": false,
  "cudnn_benchmark": false,
  "cudnn_deterministic": false,
  "cuda_matmul_allow_tf32": false,
  "cudnn_allow_tf32": true,
  "float32_matmul_precision": "highest",
  "cublas_workspace_config": ":4096:8",
  "historical_exact_replay_gpu": "NVIDIA A100-SXM4-80GB",
  "historical_exact_replay_cudnn_version": 91900
}
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Replayin

## 4. Pobierz wynik — także wtedy, gdy kontrola zgłosiła błąd

ZIP zawiera predykcje/metryki albo raport zatrzymania, log i parametry środowiska. Nie zawiera źródłowych obrazów ani wag. **Prześlij ten ZIP do oceny przed jakąkolwiek aktualizacją artykułu lub repozytorium.**


In [ ]:
result_zip = shutil.make_archive(str(work / ("Diagnostics_shape_GPU_result_" + stamp)), "zip", root_dir=output)
root_result = Path("/content") / Path(result_zip).name
shutil.copy2(result_zip, root_result)
print("Plik:", root_result.name, "bytes:", root_result.stat().st_size, "SHA256:", file_sha256(root_result))
print((output / "baseline_replay_gate.json").read_text())


Plik: Diagnostics_shape_GPU_result_20261101T161103Z.zip bytes: 161243 SHA256: bf704bcedebf7784e95d7833072c26568a0a6c1a235b7042641f10a115501858
{
  "status": "PASS",
  "argmax_matches": 150,
  "correct": 138,
  "total": 150,
  "maximum_absolute_probability_difference": 0.0,
  "mean_absolute_probability_difference": 0.0,
  "prespecified_absolute_tolerance": 0.0001,
  "exact_probability_array_match": true
}

